# PolyBridge research · Trade the 8-K

Pre-registration: [HYPOTHESIS.md](HYPOTHESIS.md) (19:38 ET, 2 Oct 2026) and [HYPOTHESIS_TAGS.md](HYPOTHESIS_TAGS.md)
(20:08 ET), both committed before any event data was fetched.

Two confirmatory hypotheses, each on the top-100 US stocks, judged against a placebo of ordinary days:

- **H1 (hedge):** after a litigation, investigation, cybersecurity or impairment 8-K the stock keeps moving by more than the options priced, so the protective put beats the placebo's.
- **H2 (opportunity):** after a restructuring plan, workforce reduction, facility closure or business-line exit 8-K the stock moves by less than the options priced, so the cash-secured put beats the placebo's.

**Pass rule (decided before results).** A hypothesis passes only if the 97.5% CI of the P&L edge (events minus ordinary days) is entirely above 0 at 2 or more of 21, 42 and expiry,
and the price-gap ratio points the predicted way (H1 above ordinary days, H2 below) at those horizons. Otherwise it is a null result.

If a pairing fails, PolyBridge reports "no edge for this event and stock — try another", and that is a valid result.

**How to run.** Only `MASSIVE_API_KEY` is required (environment variable or a `.env` file). Optionally set
`SEC_USER_AGENT="Name email"` to use EDGAR acceptance times. Edit `START, END` in the next cell and rerun all cells.


In [ ]:
# ---- Window. Judges: set your sealed window here and rerun all cells. ----
START, END = "2024-01-01", "2025-12-31"
RUN_OOS = False          # flipped once, after the method freeze (see research/HYPOTHESIS.md §4)
RUN_ATLAS = False        # exploratory atlas over every 8-K tag (HYPOTHESIS.md §5); stretch goal, slow
ATLAS_MAX_EVENTS = 15
MAX_WORKERS = 8

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from polybridge_research.analysis import decay_table, difference_board, pass_check, sample_placebo, scoreboard
from polybridge_research.atlas import count_variants, run_atlas
from polybridge_research.calendar import TradingCalendar
from polybridge_research.config import StudyConfig
from polybridge_research.costs import cost_table
from polybridge_research.evaluate import evaluate
from polybridge_research.massive import MassiveClient, load_api_key
from polybridge_research.pipeline import run_family_study
from polybridge_research.pricing import price_events

cfg = StudyConfig()
cfg.validate()


In [ ]:
from IPython.display import Markdown, display

key = load_api_key(search_from=Path.cwd())
client = MassiveClient(key)
cal = TradingCalendar()
LAST = cal.last_completed()
UA = os.environ.get("SEC_USER_AGENT")
FAMILIES = ["hedge", "opportunity"]
HEADS = list(cfg.headline_horizons)
print("Timing mode:", "EDGAR acceptance times" if UA else "conservative (every filing treated as public after the close)")
print(f"Window {START} -> {END}; last completed session {LAST.date()}")


def show(df, title=None):
    """Display a frame, or say so when there is nothing to show."""
    if title:
        print(title)
    if df is None or len(df) == 0:
        print("no events in this window")
    else:
        display(df)


def of_family(df, fam):
    if df is None or len(df) == 0 or "family" not in df:
        return pd.DataFrame() if df is None else df.iloc[0:0]
    return df[df["family"] == fam]


def hpos(horizons):
    order = [*cfg.horizons, "exp"]
    return [order.index(h) for h in horizons]


In [ ]:
study = run_family_study(client, cal, cfg, START, END, LAST, user_agent=UA, max_workers=MAX_WORKERS)
ev = study["events"]
if ev.empty:
    print("no events in this window")
else:
    by_year = ev.assign(year=pd.to_datetime(ev["filing_date"]).dt.year).groupby(["family", "year"]).size().unstack(fill_value=0)
    show(by_year, "Events by family and year")
print("Filings excluded because they fit neither family (cross-family or other tags):", len(study["excluded"]))
print("Timing counts:", study["timing_counts"])
dropped = study["dropped"]
if len(dropped):
    show(dropped.groupby("reason").size().rename("n").to_frame(), "Events dropped before pricing, by reason")
else:
    print("no events dropped")


In [ ]:
res, pl = study["results"], study["placebo_results"]
rows_f = [f for f in FAMILIES if len(of_family(res, f))]
if not rows_f:
    print("no events in this window")
else:
    fig, axes = plt.subplots(len(rows_f), 6, figsize=(22, 3.6 * len(rows_f)), squeeze=False, sharex=True)
    for i, fam in enumerate(rows_f):
        ev_r, pl_r = of_family(res, fam), of_family(pl, fam)
        sb_e = scoreboard(ev_r, cfg)
        show(sb_e, f"[{fam}] events: mean P&L per $1 of spot (95% CI)")
        if len(pl_r):
            sb_p = scoreboard(pl_r, cfg)
            show(sb_p, f"[{fam}] placebo")
            show(difference_board(ev_r, pl_r, cfg, level=cfg.confirmatory_level),
                 f"[{fam}] events minus placebo (97.5% CI)")
        else:
            sb_p = None
            print(f"[{fam}] no placebo days priced")
        for j, s in enumerate(sb_e["strategy"].unique()):
            ax = axes[i][j]
            for sb, label, colr in ((sb_e, "events", "tab:red"), (sb_p, "placebo", "tab:gray")):
                if sb is None:
                    continue
                d = sb[sb.strategy == s].dropna(subset=["mean"])
                if d.empty:
                    continue
                x = hpos(d["horizon"])
                ax.plot(x, d["mean"], marker="o", color=colr, label=label)
                ax.fill_between(x, d["ci_lo"], d["ci_hi"], color=colr, alpha=0.2)
            ax.axhline(0, color="k", lw=0.5)
            ax.set_title(f"{fam}: {s}", fontsize=9)
            ax.set_xticks(range(len(cfg.horizons) + 1))
            ax.set_xticklabels([*map(str, cfg.horizons), "exp"], fontsize=7)
        axes[i][0].legend(fontsize=7)
    fig.suptitle("Mean P&L per $1 of spot by horizon: events vs placebo (95% CI band)")
    plt.tight_layout()
    plt.show()


In [ ]:
for fam in FAMILIES:
    chk = study["checks"].get(fam)
    if chk is None:
        print(f"{fam}: no events in this window")
        continue
    pnl, ratio = chk["pnl"], chk["ratio"]
    if pnl.empty:
        print(f"{fam}: no headline horizons available in this window")
    else:
        t = pnl[["horizon", "n_a", "n_b", "difference", "ci_lo", "ci_hi"]].rename(
            columns={"n_a": "n_events", "n_b": "n_placebo", "difference": "pnl_edge", "ci_lo": "pnl_ci_lo", "ci_hi": "pnl_ci_hi"})
        t = t.merge(ratio[["horizon", "difference"]].rename(columns={"difference": "ratio_diff"}), on="horizon", how="left")
        t["pnl_ok"] = t["horizon"].isin(chk["horizons_pnl_ok"])
        t["ratio_ok"] = t["horizon"].isin(chk["horizons_ratio_ok"])
        show(t.set_index("horizon"), f"{fam} ({chk['strategy']}): events minus placebo at the headline horizons")
    verdict = "PASS" if chk["passed"] else "NULL"
    display(Markdown(f"**{fam.upper()}: {verdict}**. PASS rule: the 97.5% CI of the P&L edge (events minus ordinary days) is entirely above 0, "
                     "and the price-gap ratio points the predicted way (H1 above ordinary days, H2 below), "
                     "at 2 or more of 21, 42 and expiry."))


In [ ]:
decay = {}
for fam in FAMILIES:
    de = decay_table(of_family(study["results"], fam), cfg)
    dp = decay_table(of_family(study["placebo_results"], fam), cfg)
    decay[fam] = (de, dp)
    show(de, f"[{fam}] parity decay, events (ratio = realized / priced, entry on the pre-event session)")
    show(dp, f"[{fam}] parity decay, placebo")
if all(de.empty for de, _ in decay.values()):
    print("no events in this window")
else:
    fig, axes = plt.subplots(1, len(FAMILIES), figsize=(12, 3.8), squeeze=False)
    for ax, fam in zip(axes[0], FAMILIES):
        for tbl, label, colr in ((decay[fam][0], "events", "tab:red"), (decay[fam][1], "placebo", "tab:gray")):
            d = tbl.dropna(subset=["mean_ratio"])
            if d.empty:
                continue
            x = hpos(d.index)
            ax.plot(x, d["mean_ratio"], marker="o", color=colr, label=label)
            ax.fill_between(x, d["ci_lo"], d["ci_hi"], color=colr, alpha=0.2)
        ax.axhline(1, color="k", lw=0.5, ls="--")
        ax.set_title(f"{fam}: mean parity ratio by horizon")
        ax.set_xticks(range(len(cfg.horizons) + 1))
        ax.set_xticklabels([*map(str, cfg.horizons), "exp"])
        ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
from polybridge_research.schema import STRATEGY_FOR_FAMILY, Family

for fam in FAMILIES:
    strat = STRATEGY_FOR_FAMILY[Family(fam)]
    r = of_family(study["results"], fam)
    r = r[r["horizon"] == 21] if len(r) else r
    if len(r):
        grid = r.groupby(["bucket", "entry", "otm"])[strat].agg(n="count", mean_pnl="mean")
        show(grid, f"[{fam}] {strat}: mean P&L at h=21 by bucket x entry x OTM (the starter's grid; entry 'pre' is a pricing statement, not a trade)")
    else:
        print(f"[{fam}] no events in this window")
print("Variants in this sensitivity grid (one tag):", count_variants(cfg))


In [ ]:
cost_summ = []
QUOTES_OK = True
for fam in FAMILIES:
    strat = STRATEGY_FOR_FAMILY[Family(fam)]
    res_f = of_family(study["results"], fam)
    priced_f = [p for p in study["priced"] if str(getattr(p.family, "value", p.family)) == fam]
    for h in (21, 42):
        if res_f.empty or not priced_f:
            print(f"[{fam}] h={h}: no events in this window")
            continue
        ct = None
        if QUOTES_OK:
            try:
                ct = cost_table(res_f, priced_f, strat, h, cfg, client=client)
            except Exception:
                QUOTES_OK = False
                print("Options quotes unavailable on this key; spread cost not computed (haircut costs shown).")
        if ct is None:
            ct = cost_table(res_f, priced_f, strat, h, cfg, client=None)
        if ct.empty:
            print(f"[{fam}] h={h}: no events in this window")
            continue
        cost_summ.append({"family": fam, "strategy": strat, "horizon": h, "n": len(ct),
                          "gross": ct["gross"].mean(), "net_haircut_1x": ct["net_haircut_1x"].mean(),
                          "net_haircut_2x": ct["net_haircut_2x"].mean(), "spread_cost": ct["spread_cost"].mean(),
                          "net_spread": ct["net_spread"].mean(), "median_leg_volume": ct["leg_volume"].median()})
if cost_summ:
    show(pd.DataFrame(cost_summ).set_index(["family", "horizon"]),
         "Costs: mean P&L per $1 of spot, gross vs net (5% premium haircut 1x/2x; real half-spreads where quotes exist)")


In [ ]:
if RUN_ATLAS:
    rows = client.get_all("/stocks/taxonomies/vX/disclosures", {"limit": 1000})
    tags = sorted({t["tertiary_category"] for t in rows})
    base = {cfg.baseline_bucket: cfg.buckets[cfg.baseline_bucket]}
    atlas_pl_events = sample_placebo(pd.DataFrame({"ticker": list(cfg.universe),
                                                   "filing_date": [pd.Timestamp("1900-01-01")] * len(cfg.universe)}),
                                     300, START, END, cal, gap_days=0, seed=11)
    atlas_pl_priced, _ = price_events(client, atlas_pl_events, cal, cfg, buckets=base, max_workers=MAX_WORKERS,
                                      label="atlas placebo")
    atlas_pl_res = evaluate(atlas_pl_priced, cal, cfg, LAST)
    atlas = run_atlas(client, cal, cfg, tags, START, END, LAST, atlas_pl_res,
                      max_events_per_tag=ATLAS_MAX_EVENTS, max_workers=MAX_WORKERS)
    print("EXPLORATORY: the 15 lowest q-values across tags. Nothing here is a headline (HYPOTHESIS.md §5).")
    show(atlas.sort_values("q_value").head(15))
    print("Variants counted (all tags, full grid):", count_variants(cfg, n_tags=len(tags)))
else:
    print("Atlas not run (RUN_ATLAS = False). It is exploratory and slow; set RUN_ATLAS = True to include it.")


In [ ]:
if RUN_OOS:
    oos = run_family_study(client, cal, cfg, cfg.oos_start, cfg.oos_end, LAST, user_agent=UA, max_workers=MAX_WORKERS)
    for fam, chk in oos["checks"].items():
        ins = study["checks"].get(fam)
        if ins is None:
            print(f"{fam}: no in-sample events to compare")
            continue
        merged = ins["pnl"][["horizon", "difference"]].merge(chk["pnl"][["horizon", "difference"]], on="horizon",
                                                             suffixes=("_in", "_oos"))
        merged["same_sign"] = np.sign(merged.difference_in) == np.sign(merged.difference_oos)
        print(f"{fam}: in-sample -> out-of-sample edge of {chk['strategy']} (events minus placebo)")
        display(merged)
else:
    print("Out-of-sample not run: it runs once, after the method freeze.")


## Sealed window for judges

Set `START` / `END` in the configuration cell to your window and rerun all cells. Nothing else is needed:
the study, placebo, pass check, decay, sensitivity and costs all follow the window.


In [ ]:
print(f"This run covered {START} -> {END}. Events: {len(study['events'])}; priced (event, bucket) pairs: {len(study['priced'])}.")


## Known limitations

- **Static universe.** `TOP_100` is today's list applied to the whole window: a survivorship bias.
- **Spot is inferred.** The stock price comes from put-call parity on the ATM pair with a flat carry rate and no dividend term; both legs are last trades and can be stale. The error largely cancels in P&L because entry and exit use the same construction.
- **The synthetic stock is not shares.** It collects no dividends and its short put can be assigned early; neither is modelled.
- **Last-trade marks, no spreads.** P&L is theoretical, mid-market at best. The cost section applies a 5% premium haircut (1x and 2x) and real half-spreads where quotes exist.
- **Small samples.** Intervals are wide; read the `n` columns before the means.
- **No earnings flag.** Events that share their window with an earnings release are not separated.
- **Filing lag and time of day.** `filing_date` has no time of day. With `SEC_USER_AGENT` set, EDGAR acceptance times decide the first tradable session. **Without it, every filing is treated as public after the close** (the next session is `t_0`): conservative, no lookahead, and slightly late for pre-market filings.
- **Strike availability.** When no strike sits at the requested OTM distance, the nearest listed one is used.
- **Exploratory material** (the atlas) never replaces H1 or H2.
